In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

# Imports

In [5]:
from harp.data.collect import collect
from harp.data.process import clean_df, group_by_reviewer, inject_rejected, encode
from harp.data.cutoffs import calculate_cutoffs

import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Data Pipeline

In [26]:
collect(5, "harp/data/raw/samples")
clean_df(pd.read_csv("harp/data/raw/cms_2008_2010_samples.csv"))

df = pd.read_csv("harp/data/raw/cms_2008_2010_samples.csv")
RANDOM_STATE = 42

df_train_val, df_test = train_test_split(
    df, 
    test_size=0.15, 
    random_state=RANDOM_STATE,
    shuffle=True
)

validation_size = 0.15 / 0.85
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=validation_size, 
    random_state=RANDOM_STATE,
    shuffle=True
)

df_train = inject_rejected(df_train, 0.2)
df_val = inject_rejected(df_val, 0.2)
df_test = inject_rejected(df_test, 0.2)

scores = {"multi_payer":3, "short_stay":3, "surgical":2, "high_cost":1}
cutoffs = calculate_cutoffs(
    df = pd.concat([df_train, df_val, df_test], axis=0, ignore_index=True),
    scores = scores,
    n_reviewers = 2,
    n_init = 10,
    random_state = 42,
    plot = False
)

df_train = group_by_reviewer(df_train, cutoffs, scores)
df_val = group_by_reviewer(df_val, cutoffs, scores)
df_test = group_by_reviewer(df_test, cutoffs, scores)

os.makedirs("harp/data/raw/harp_dataset")
df_train.to_csv("harp/data/raw/harp_dataset/train.csv", index=False)
df_val.to_csv("harp/data/raw/harp_dataset/val.csv", index=False)
df_test.to_csv("harp/data/raw/harp_dataset/test.csv", index=False)

os.makedirs("harp/data/raw/harp_dataset_encoded")
df_train_encoded = encode(df_train, os.path.join("harp/data/raw/harp_dataset_encoded", "train.csv"))
df_val_encoded = encode(df_val, os.path.join("harp/data/raw/harp_dataset_encoded", "val.csv"))
df_test_encoded = encode(df_test, os.path.join("harp/data/raw/harp_dataset_encoded", "test.csv"))


--- Data Per-Reviewer Distribution ---
Reviewer 1 (Score 0.0 - 1.41): 129493 claims
Reviewer 2 (Score 1.41 - 10.0): 269633 claims
--------------------------------------

